# TR3D — Eikonal Weight Ablation

Sweeps the eikonal loss weight over six values and three random seeds
(18 training runs) to quantify how sensitive the method is to this
hyperparameter, and to select the value used in `02_train_and_evaluate.ipynb`.

**Runtime:** ~3.3 h on a T4 (18 runs x ~11 min). Results are appended to CSV
after every run and completed runs are skipped on restart, so a disconnect
costs at most one run.

**Why the control matters:** `w_eik = 0` removes the signed-distance
constraint entirely. Under rigid translation the exact solution stays an exact
SDF for all time, so the eikonal term is consistent with the transport
equation here — this benchmark is favourable to it, which is worth stating
when generalising the selected value to deformation cases.

**Outputs:** `TR3D_eikonal_ablation.csv` (per-snapshot and averaged metrics for
every run) and a summary table with seed spread.

In [1]:
# --- setup ---
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/TR3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import tr3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Mounted at /content/drive
device: cuda


In [2]:
# --- configuration ---
W_EIK_GRID   = [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
SEEDS        = [42, 43, 44]

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 10000, 5000
LAYERS, NEURONS = 8, 256
T_FINAL      = 5.0
ADAM_EPOCHS  = 20000        # same budget as the reported run, so the ranking
LBFGS_EPOCHS = 500          # transfers without extrapolation
GRID_N       = 201
N_SNAPSHOTS  = 6

CSV_PATH = f'{PROJECT_DIR}/TR3D_eikonal_ablation.csv'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)
METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs']

FIELDS = ['w_eik', 'seed']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['loss_adam', 'loss_final', 'lbfgs_gain', 'minutes']

print(f'{len(W_EIK_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

18 runs, 41 columns per row


In [3]:
# --- one run ---
def run_one(w_eik, seed):
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    loss_fn = core.make_loss(net, X_f, X_i, T_FINAL, W_PDE, W_IC, w_eik)

    tr = core.train(net, loss_fn, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)
    res = core.evaluate(net, grid, times, T_FINAL, device, cache=False)

    row = {'w_eik': w_eik, 'seed': seed}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({'loss_adam': f"{tr['loss_adam']:.6e}",
                'loss_final': f"{tr['loss_final']:.6e}",
                'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
                'minutes': f"{tr['minutes']:.2f}"})

    del net, X_f, X_i
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) ---
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['w_eik']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(W_EIK_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * 11 / 60:.1f} h\n', flush=True)

for k, (wv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] w_eik={wv:g} seed={sd} ...', flush=True)
    row = run_one(wv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"eik={float(row['avg_eik_dev']):.3e}  "
          f"lbfgs_gain={float(row['lbfgs_gain']):.2e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

18 run(s) to go, ~3.3 h

[1/18] w_eik=0 seed=42 ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    relL2=14.7776%  MAPE=3.5611%  eik=7.117e-02  lbfgs_gain=1.47e-08  (12.7 min)
[2/18] w_eik=0 seed=43 ...
    relL2=7.3264%  MAPE=6.4431%  eik=2.983e-02  lbfgs_gain=7.42e-06  (13.0 min)
[3/18] w_eik=0 seed=44 ...
    relL2=13.7382%  MAPE=1.9408%  eik=9.431e-02  lbfgs_gain=3.79e-09  (13.0 min)
[4/18] w_eik=0.0001 seed=42 ...
    relL2=10.5773%  MAPE=2.7009%  eik=6.826e-02  lbfgs_gain=2.99e-09  (12.9 min)
[5/18] w_eik=0.0001 seed=43 ...
    relL2=2.7415%  MAPE=6.3202%  eik=4.374e-02  lbfgs_gain=2.32e-06  (13.0 min)
[6/18] w_eik=0.0001 seed=44 ...
    relL2=18.8680%  MAPE=5.2414%  eik=1.070e-01  lbfgs_gain=3.17e-08  (13.0 min)
[7/18] w_eik=0.001 seed=42 ...
    relL2=18.4260%  MAPE=26.7352%  eik=6.616e-02  lbfgs_gain=2.04e-08  (13.0 min)
[8/18] w_eik=0.001 seed=43 ...
    relL2=2.8470%  MAPE=0.8350%  eik=1.537e-01  lbfgs_gain=1.32e-07  (13.0 min)
[9/18] w_eik=0.001 seed=44 ...
    relL2=20.1904%  MAPE=33.6639%  eik=9.903e-02  lbfgs_gain=1.30e-08  (13.0 min)
[10/18] w_eik=0.01 seed=42 ..

## Summary and selection

Decision rule, fixed before the sweep:

1. **Primary** — lowest average relative L₂
2. **Tiebreak 1** — lowest average volume MAPE
3. **Tiebreak 2** — lowest eikonal deviation
4. **Tiebreak 3** — the smaller weight

Two weights count as *tied* when the gap between their means is smaller than
the larger of their two seed standard deviations. Eikonal deviation is only a
second-order tiebreak on purpose: it is essentially the quantity the swept term
minimises, so ranking by it would recover the sweep by construction.

In [5]:
# --- summary ---
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['w_eik']), []).append(r)

stats = {}
print(f"{'w_eik':>8} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'eik dev':>11} | {'n':>2}")
print('-' * 74)
for wv in sorted(agg):
    g = agg[wv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    stats[wv] = (m, s, len(g))
    print(f"{wv:>8g} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{m['eik_dev']:>11.3e} | {len(g):>2}")

def tied(a, b, key):
    (ma, sa, _), (mb, sb, _) = stats[a], stats[b]
    return abs(ma[key] - mb[key]) < max(sa[key], sb[key])

best = min(stats, key=lambda w: stats[w][0]['rel_L2'])
for cand in sorted(stats, key=lambda w: stats[w][0]['rel_L2'])[1:]:
    if tied(best, cand, 'rel_L2'):
        if not tied(best, cand, 'vol_MAPE'):
            best = min(best, cand, key=lambda w: stats[w][0]['vol_MAPE'])
        elif not tied(best, cand, 'eik_dev'):
            best = min(best, cand, key=lambda w: stats[w][0]['eik_dev'])
        else:
            best = min(best, cand)

ties = sorted(w for w in stats if w != best and tied(best, w, 'rel_L2'))
print('-' * 74)
print(f'selected w_eik = {best:g}')
if ties:
    print('statistically tied on rel L2 with: ' + ', '.join(f'{w:g}' for w in ties))
    print('-> report insensitivity across this range rather than a single value')
print(f"mean L-BFGS gain over all runs: "
      f"{np.mean([float(r['lbfgs_gain']) for r in rows]):.3e}")

   w_eik |          rel L2 (%) |        vol MAPE (%) |     eik dev |  n
--------------------------------------------------------------------------
       0 |   11.9474 ± 3.2950  |    3.9817 ± 1.8620  |   6.510e-02 |  3
  0.0001 |   10.7289 ± 6.5845  |    4.7542 ± 1.5172  |   7.300e-02 |  3
   0.001 |   13.8212 ± 7.7932  |   20.4114 ± 14.1286 |   1.063e-01 |  3
    0.01 |    5.8291 ± 6.9602  |   12.8033 ± 9.5253  |   2.943e-02 |  3
     0.1 |    0.4420 ± 0.2271  |    1.2073 ± 0.2402  |   2.639e-02 |  3
       1 |    1.4223 ± 0.3992  |    0.8104 ± 0.5681  |   1.204e-02 |  3
--------------------------------------------------------------------------
selected w_eik = 0.1
statistically tied on rel L2 with: 0.01
-> report insensitivity across this range rather than a single value
mean L-BFGS gain over all runs: 5.696e-07


## Error growth per snapshot

Per-snapshot averages across seeds, for the time-evolution table/figure.

In [6]:
# --- per-snapshot table ---
METRIC = 'rel_L2'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

rel_L2 by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.4109 |    1.0737 |    4.4681 |   11.5812 |   21.5853 |   32.5651
  0.0001 |    0.4416 |    1.1358 |    4.3815 |   10.7351 |   19.2807 |   28.3988
   0.001 |    0.5263 |    1.4753 |    5.7082 |   13.9417 |   24.8992 |   36.3763
    0.01 |    0.3354 |    0.7427 |    2.5096 |    5.9034 |   10.3650 |   15.1182
     0.1 |    0.1929 |    0.2567 |    0.3747 |    0.5058 |    0.6111 |    0.7108
       1 |    0.2016 |    0.4501 |    0.9406 |    1.6028 |    2.3228 |    3.0162


In [7]:
# --- per-snapshot table ---
METRIC = 'vol_MAPE'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

vol_MAPE by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    3.6205 |    3.7560 |    3.8256 |    3.8135 |    3.7257 |    5.1487
  0.0001 |    2.9113 |    3.0952 |    3.1718 |    3.1565 |    3.0300 |   13.1603
   0.001 |    1.2074 |    1.0589 |    1.0512 |    1.1718 |   12.5124 |  105.4666
    0.01 |    4.6585 |    5.3093 |    5.7216 |    7.2578 |   16.8910 |   36.9814
     0.1 |    1.0777 |    1.1803 |    1.2652 |    1.2814 |    1.2463 |    1.1930
       1 |    0.5248 |    0.7164 |    0.9514 |    0.9673 |    0.8888 |    0.8139


In [8]:
# --- per-snapshot table ---
METRIC = 'eik_dev'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

eik_dev by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0652 |    0.0650 |    0.0649 |    0.0649 |    0.0651 |    0.0655
  0.0001 |    0.0731 |    0.0726 |    0.0725 |    0.0727 |    0.0732 |    0.0738
   0.001 |    0.1067 |    0.1061 |    0.1059 |    0.1060 |    0.1063 |    0.1068
    0.01 |    0.0304 |    0.0299 |    0.0295 |    0.0291 |    0.0289 |    0.0288
     0.1 |    0.0249 |    0.0253 |    0.0258 |    0.0265 |    0.0274 |    0.0285
       1 |    0.0128 |    0.0117 |    0.0114 |    0.0115 |    0.0120 |    0.0129


In [9]:
# --- per-snapshot table ---
METRIC = 'l2_abs'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

l2_abs by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0018 |    0.0043 |    0.0166 |    0.0418 |    0.0801 |    0.1306
  0.0001 |    0.0020 |    0.0046 |    0.0163 |    0.0387 |    0.0716 |    0.1139
   0.001 |    0.0023 |    0.0059 |    0.0212 |    0.0503 |    0.0924 |    0.1459
    0.01 |    0.0015 |    0.0030 |    0.0093 |    0.0213 |    0.0385 |    0.0606
     0.1 |    0.0009 |    0.0010 |    0.0014 |    0.0018 |    0.0023 |    0.0029
       1 |    0.0009 |    0.0018 |    0.0035 |    0.0058 |    0.0086 |    0.0121
